# M50 · The focal sphere

**Part 3 · week 6 · about 9 h core**

## 한국어 요약

이 모듈에서는 단층의 방향과 미끄러짐을 주향, 경사, 미끄럼각으로 기술하고, 이를 Aki & Richards의 북-동-아래 좌표계에서 단층면 법선 벡터와 미끄러짐 벡터로 바꾸는 함수를 직접 작성합니다. 이 두 벡터로 보조면과 P축, T축을 계산해 ObsPy와 Pyrocko로 검증하고, 지진파 파선이 진원을 떠나는 방향을 하반구 등면적 투영(슈미트망)에 그리는 함수를 만듭니다. 이어서 M30의 파선 추적으로 이탈각을 계산하고, 2019년 상주 지진(ML 3.9)에서 M20에 직접 읽은 초동 극성을 진원구 위에 올려 한국지질자원연구원(KIGAM) 지진연보의 단층면해와 비교합니다. 마지막으로 실무 도구인 ObsPy의 비치볼을 같은 조건으로 다시 그린 뒤 단층면 하나를 바꾸어 그 결과를 설명합니다. 코드를 쓰기 전에 NMSOP-2의 손 연습 문제를 인쇄한 등면적망 위에서 먼저 풀어 봅니다.

## What you will be able to do

- Turn strike, dip and rake into the fault normal and the slip vector in north-east-down coordinates (Aki & Richards), and say how they differ from the up-south-east order of the Global CMT catalog and ObsPy
- Compute the auxiliary plane and the P and T axes of a mechanism from those two vectors, and test them against ObsPy and Pyrocko
- Plot rays on a lower-hemisphere equal-area net from their takeoff angle and azimuth, with upward rays drawn at the opposite point, and say why the net keeps areas
- Compute takeoff angles with your own ray tracing of M30, including the first arrivals that are head waves, and plot your Sangju first motions against a published mechanism
- Draw the mechanism with ObsPy as a production tool, unchanged and with the other nodal plane, and explain why first motions alone cannot tell the fault plane

## Before you start

- Modules: M20, M30, M31
- Reading: Bormann, Baumbach & Wendt (2012), the whole exercise, by hand on the printed net, before this module; Bormann (2012), the whole exercise, by hand, before Part 5; Stein & Wysession (2003), the section on focal mechanisms; Aki & Richards (2002), the section on the geometry of a fault (strike, dip, rake) in chapter 4
- Needs: synthetic, stp
- Key terms (see the glossary): fault (단층), strike (주향), dip (경사), rake (미끄럼각, 레이크), hanging wall (상반), footwall (하반), normal fault (정단층), reverse fault (역단층), strike-slip fault (주향이동단층), unit vector (단위 벡터), dot product (내적 (스칼라곱)), cross product (외적 (벡터곱)), fault normal (단층면 법선 벡터), slip vector (미끄러짐 벡터), north-east-down coordinates (NED; 북-동-아래 좌표계), up-south-east coordinates (USE; 위-남-동 좌표계), moment tensor (모멘트 텐서), double couple (이중 우력, 더블 커플), focal mechanism (단층면해, 진원 메커니즘), nodal plane (절면), auxiliary plane (보조면), P and T axes (P축과 T축), null axis (B axis; 영축 (B축)), trend (트렌드 (방위각)), plunge (플런지 (침강각)), focal sphere (진원구), takeoff angle (이탈각), azimuth (방위각), upgoing ray (상향 파선), opposite direction (antipode; 대척 방향 (대척점)), lower-hemisphere projection (하반구 투영), equal-area projection (Schmidt net; 등면적 투영 (슈미트망)), equal-angle projection (Wulff net; 등각 투영 (울프망)), stereonet (스테레오넷, 평사투영망), radiation pattern (방사 패턴), compression (first motion; 압축 (초동)), dilatation (first motion; 팽창 (초동)), first-motion polarity (초동 극성), polarity reversal (극성 반전), first arrival (최초 도달파 (초동)), head wave (선두파), crossover distance (교차 거리), critical angle (임계각), ray parameter (파선 변수), beachball (비치볼, 단층면해 그림), checksum (SHA-256 fingerprint; 체크섬), widget (위젯)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M50.yaml](../../../.sgtl/criteria/M50.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M50.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # folder of your STP download (module M03) inside the data folder
KMA_EVENT_ID = "2019009762"               # the ML 3.9 Sangju earthquake of 21 July 2019 in the KMA catalog
PICKS_CSV = "../picks/M20_manual_picks.csv"   # your manual picks of module M20 (in work/picks/)
M31_FILE = "../M31_location/results/M31_hypoinverse_runs.csv"   # your location table of module M31 (task M31-T6)
M31_ROW = "your geiger"                   # the row of that table with your own location of the mainshock
MODEL_NAME = "kim1983"                    # the velocity model of your takeoff angles (Parts 5 and 6)
PRACTICE_PLANE = (30.0, 60.0, -90.0)      # a made-up fault (strike, dip, rake in degrees) for Parts 1-4
CHANGED_PLANE = "B"                       # task 6: the one input you change in ObsPy's beachball (plane A -> B)
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M50")

import hashlib
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, read
from obspy.geodetics import gps2dist_azimuth

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M50-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up faults and rays whose answers come from two
other programs (ObsPy and Pyrocko, which your functions may not use) or from how the cases were built
(`st.reroll("M50-T1")` draws new ones); in task 6 they check your own Sangju files and never print a value.

The order of the work: first you write the geometry of a fault and of the focal sphere and test it on cases with
a known answer (tasks 1-5), then you apply it to the first motions you picked and compare them with a published
mechanism (task 6), and last you draw the same mechanism with a production tool, ObsPy's beachball, once
unchanged and once with one input changed.

**By hand first.** Before coding the focal-sphere tasks, print the blank equal-area net of the course,
[docs/figures/equal_area_net.pdf](../../../docs/figures/equal_area_net.pdf) (radius 10 cm on A4; no scaling when
you print), and do two exercises of the *New Manual of Seismological Observatory Practice* (NMSOP-2) on it with
tracing paper and a pin through the centre: EX 3.2, *Determination of fault-plane solutions by hand* (Bormann,
Baumbach & Wendt 2012, [doi:10.2312/GFZ.NMSOP-2_EX_3.2](https://doi.org/10.2312/GFZ.NMSOP-2_EX_3.2)), and EX 3.3
on takeoff angles (Bormann 2012, [doi:10.2312/GFZ.NMSOP-2_EX_3.3](https://doi.org/10.2312/GFZ.NMSOP-2_EX_3.3)),
before Part 5. Both are open PDFs; keep your sheets, task 4 asks about them. The SERC *Focal mechanism primer*
(reference `serc_focal_primer`) is a short illustrated introduction if the words below are new to you.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M50.md`.

## Part 1 · Strike, dip, rake and two vectors (about 30 min, then task 1)

A **fault** (glossary `fault`) is a surface in the rock along which the two sides slipped. Near an earthquake
it is flat enough to be a plane, and three angles describe the plane and the slip on it (Aki & Richards 2002,
chapter 4; Stein & Wysession 2003):

- **strike** (glossary `strike`), φ: the compass direction of a horizontal line in the plane, clockwise from
  north, 0 to 360 degrees. Of the two opposite directions of that line, the strike is the one with the plane
  dipping to its right. So the **dip direction** is φ + 90.
- **dip** (glossary `dip`), δ: the angle of the plane below the horizontal, 0 to 90 degrees.
- **rake** (glossary `rake`), λ: the direction of slip inside the plane, from the strike direction, −180 to 180
  degrees. It is the motion of the **hanging wall** (glossary `hanging_wall`), the block above the plane,
  relative to the **footwall** (glossary `footwall`), the block below it. A positive rake moves the hanging wall
  up: λ = 90 is a pure **reverse fault** (glossary `reverse_fault`), λ = −90 a pure **normal fault** (glossary
  `normal_fault`), and λ = 0 and 180 are **strike-slip faults** (glossary `strike_slip_fault`), left-lateral and
  right-lateral.

Computers work with vectors instead. A **unit vector** (glossary `unit_vector`) has length 1 and gives a
direction only. Two of them describe a fault: the **fault normal** n (glossary `fault_normal`), perpendicular to
the plane, and the **slip vector** d (glossary `slip_vector`), the direction in which the hanging wall moved. We
take the normal that points up, out of the footwall into the hanging wall. The slip lies in the plane, so the
**dot product** (glossary `dot_product`) n · d = n₁d₁ + n₂d₂ + n₃d₃, which is zero for perpendicular vectors, is
zero.

**Which coordinates?** This module uses the order of Aki & Richards, **north-east-down** (glossary
`north_east_down`, NED): x₁ points north, x₂ east, x₃ down. The Global CMT catalog and ObsPy's beachball
functions use **up-south-east** (glossary `up_south_east`, USE): r up, θ south, φ east. The same vector has
different numbers in the two orders, (north, east, down) = (−θ, φ, −r); mixing them up turns a mechanism into a
different one without any error message. Every vector in this module is in north-east-down, unless a cell says
otherwise.

**What the check of task 1 compares with.** Seen from far away, the forces that an earthquake sets off at its
source are described by a **moment tensor** (glossary `moment_tensor`): a symmetric 3-by-3 matrix M (Aki &
Richards 2002, chapter 3; Jost & Herrmann 1989, reference `jost1989`, a short introduction). Slip on a flat fault
acts like two pairs of opposite forces with no net force and no net turning, a **double couple** (glossary
`double_couple`). For a fault with unit slip on unit area (and unit rigidity), its moment tensor is
M = n dᵀ + d nᵀ, the matrix whose entry in row j and column k is n_j d_k + d_j n_k. Pyrocko builds M from
strike, dip and rake without your vectors, so it is an independent test of them; module M52 derives the formula.

### M50-T1 · Fault normal and slip vectors · core · about 60 min

**Goal:** write `fault_vectors(strike, dip, rake)`: the unit normal and the unit slip vector of a fault, in
north-east-down coordinates.

**Given:** the definitions of Part 1 and numpy. Not allowed in this function: all of ObsPy's imaging package
(`obspy.imaging`, with its beachball, source and MoPaD code), Pyrocko, MTUQ, MoPaD, the mechanism code of
SKHASH and PocketQuake, and the course package (the check compares with ObsPy and Pyrocko).

**Contract:**
- Inputs: `strike`, `dip`, `rake` in degrees (numbers; 0 ≤ dip ≤ 90).
- Output: a tuple `(normal, slip)` of two numpy arrays of shape (3,): (north, east, down) components, each of
  length 1. `normal` points up (its down component is ≤ 0), into the hanging wall; `slip` is the motion of the
  hanging wall relative to the footwall.
- Example: a vertical fault striking north with rake 0 (strike 0, dip 90, rake 0, left-lateral) gives the
  normal (0, 1, 0), east, and the slip (1, 0, 0), north.

**Done when** `st.check("M50-T1", func=fault_vectors)` passes on five fault types and 40 random planes: the
moment tensor that Pyrocko builds from the same three angles must equal n dᵀ + d nᵀ from your two vectors (a
double couple has exactly that form; module M52 explains why), ObsPy must read the strike and dip of the plane
back from your normal, and the normal must point up.

**Stuck?** `st.hint("M50-T1", 1)`

In [ ]:
# Before you code: what kind of fault is PRACTICE_PLANE (strike 30, dip 60, rake -90)? To which side does it dip,
# and which way does its hanging wall move? Will the down components of your normal and of your slip be positive,
# negative or zero? Write down your guesses with a reason.
st.predict("M50-T1", """

""")

In [ ]:
def fault_vectors(strike, dip, rake):
    """Unit fault normal and unit slip vector of a fault, north-east-down (Aki & Richards).

    strike, dip, rake : degrees (strike clockwise from north, the plane dipping to its right; 0 <= dip <= 90;
                        rake from the strike direction, positive when the hanging wall moves up)
    returns           : (normal, slip), two numpy arrays of shape (3,) with (north, east, down) components;
                        normal points up (into the hanging wall), slip is the motion of the hanging wall
    """
    # Step 1: the angles in radians.
    # Step 2: the normal: tilted from the vertical by the dip, its horizontal part towards the dip direction
    #         (strike + 90), pointing up.
    # Step 3: the slip: cos(rake) along the strike direction plus sin(rake) up the dip (the opposite of the
    #         down-dip direction).
    # YOUR CODE HERE
    raise NotImplementedError("M50-T1: write your code here")

In [ ]:
st.check("M50-T1", func=fault_vectors)

Your vectors for the practice fault and for the five fault types of the check, and the same practice vectors in
the up-south-east order that ObsPy reads (the conversion is the line marked USE).

In [ ]:
faults = {"practice": PRACTICE_PLANE, "vertical left-lateral": (0.0, 90.0, 0.0), "thrust": (0.0, 45.0, 90.0),
          "normal": (90.0, 30.0, -90.0), "right-lateral": (210.0, 60.0, 180.0), "oblique": (135.0, 75.0, 45.0)}
rows = []
for label, plane in faults.items():
    n_vec, d_vec = fault_vectors(*plane)
    rows.append({"fault": label, "strike, dip, rake": plane, "normal (N, E, D)": np.round(n_vec, 3).tolist(),
                 "slip (N, E, D)": np.round(d_vec, 3).tolist(), "n . d": round(float(n_vec @ d_vec), 12)})
print(pd.DataFrame(rows).to_string(index=False))
n_vec, d_vec = fault_vectors(*PRACTICE_PLANE)
to_use = np.array([[0.0, 0.0, -1.0], [-1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])   # USE: up = -down, south = -north
print("\npractice fault in up-south-east: normal", np.round(to_use @ n_vec, 3).tolist(), " slip",
      np.round(to_use @ d_vec, 3).tolist())

**Explain (3-5 sentences):** Was your prediction for the practice fault right (the signs of the two down
components)? Use the table: what do the normal and the slip of the thrust and of the normal fault have in
common, and what tells the two apart? Why must a program that reads mechanisms know the coordinate order of
its vectors, and what would happen to the practice fault if you handed its north-east-down vectors to a program
that expects up-south-east?

## Part 2 · Two planes fit the same first motions (about 20 min, then task 2)

A **focal mechanism** (glossary `focal_mechanism`) is the orientation of the fault and the direction of slip of
an earthquake, as far as seismic waves tell. The P wave leaves the source with no motion along two planes, the
**nodal planes** (glossary `nodal_plane`): the fault plane, and the plane perpendicular to the slip. The second
one is the **auxiliary plane** (glossary `auxiliary_plane`). Its normal is the slip vector of the fault, and its
slip is the fault's normal: swap n and d, and the P wave is the same, because it depends on n and d only through
the sum n dᵀ + d nᵀ, which does not change when the two are swapped (Stein & Wysession 2003). That is why a
mechanism from first motions always comes as two planes, and why bulletins print both.

To turn the swapped vectors back into strike, dip and rake you read the angles off a normal and a slip: the
inverse of task 1. Mind the sign: the new normal may point down; then flip both new vectors (−n, −d describe
the same motion) before you read the angles.

### M50-T2 · The auxiliary plane · core · about 60 min

**Goal:** write `auxiliary_plane(strike, dip, rake)`: strike, dip and rake of the other nodal plane.

**Given:** your `fault_vectors` (task 1) and numpy. Not allowed in this function: the same libraries as in
task 1 (ObsPy has a function for exactly this, which the check uses).

**Contract:**
- Inputs: `strike`, `dip`, `rake` of one nodal plane, in degrees (0 < dip ≤ 90).
- Output: a tuple `(strike2, dip2, rake2)` of floats in degrees, with 0 ≤ strike2 < 360, 0 ≤ dip2 ≤ 90 and
  −180 < rake2 ≤ 180, in the conventions of Part 1.
- Example: an invented oblique fault (strike 100, dip 60, rake 30) has the other plane (353.90, 64.34, 146.31).

**Done when** `st.check("M50-T2", func=auxiliary_plane)` passes on four planes by hand and 40 random planes,
against ObsPy's `aux_plane` (to 0.01 degrees).

**Stuck?** `st.hint("M50-T2", 1)`

In [ ]:
# Before you code: by hand on your net (or by thinking in 3-D): what are strike, dip and rake of the other plane
# of a 45-degree thrust striking north (0, 45, 90)? And of the practice fault (30, 60, -90)? How many degrees apart
# do you expect the auxiliary plane of KIGAM's printed plane A (Part 6: 288, 89, 7) and KIGAM's printed plane B
# (198, 83, 179) to be, and what in the printed numbers decides the size of that angle?
st.predict("M50-T2", """

""")

In [ ]:
def auxiliary_plane(strike, dip, rake):
    """Strike, dip and rake (degrees) of the other nodal plane of a double-couple mechanism.

    strike, dip, rake : one nodal plane in degrees (Aki & Richards conventions, as in fault_vectors)
    returns           : (strike2, dip2, rake2) floats; 0 <= strike2 < 360, 0 <= dip2 <= 90, -180 < rake2 <= 180
    """
    # Step 1: the normal and slip of the given plane (task 1); the other plane has the slip as its normal and the
    #         normal as its slip.
    # Step 2: if the new normal points down, flip both new vectors.
    # Step 3: dip from the normal's down part; strike 90 degrees to the left of the normal's horizontal part
    #         (np.arctan2); rake = the angle from the new strike direction to the new slip, positive when the
    #         slip goes up.
    # YOUR CODE HERE
    raise NotImplementedError("M50-T2: write your code here")

In [ ]:
st.check("M50-T2", func=auxiliary_plane)

Your auxiliary planes of the two practice faults, and of each of KIGAM's two printed planes of the Sangju
mainshock (Part 6 quotes them with their source), next to the other printed plane. The angle in the last
column is the angle between the two normals: 0 when the planes coincide.

In [ ]:
KIGAM_A, KIGAM_B = (288.0, 89.0, 7.0), (198.0, 83.0, 179.0)    # KIGAM bulletin 2019, report KEMS-2019-062
print("thrust (0, 45, 90) ->", np.round(auxiliary_plane(0.0, 45.0, 90.0), 3))
print("practice", PRACTICE_PLANE, "->", np.round(auxiliary_plane(*PRACTICE_PLANE), 3))
for name, plane, printed in (("A", KIGAM_A, KIGAM_B), ("B", KIGAM_B, KIGAM_A)):
    mine = auxiliary_plane(*plane)
    cos_angle = abs(float(fault_vectors(*mine)[0] @ fault_vectors(*printed)[0]))
    print(f"KIGAM plane {name} {plane} -> your auxiliary plane {np.round(mine, 2).tolist()}; the other printed "
          f"plane {printed}: {np.degrees(np.arccos(min(cos_angle, 1.0))):.2f} degrees apart")

**Explain (3-5 sentences):** Were your hand answers for the thrust and the practice fault right? How far is your
auxiliary plane of KIGAM's plane A from KIGAM's printed plane B (and the other way round), and is that more or
less than whole-degree rounding of the printed angles can explain? Why can first motions of P never tell you
which of the two planes slipped, and what other information could?

## Part 3 · P and T axes (about 15 min, then task 3)

Between the two nodal planes lie the two axes of a mechanism (glossary `p_t_axes`): the **T axis**, in the
middle of the quadrants where the P wave pushes the ground away from the source, and the **P axis**, in the
middle of the quadrants where it pulls. Both lie at 45 degrees to the two planes, in the plane of n and d: T
along n + d, P along n − d (Stein & Wysession 2003). The third axis, perpendicular to both, is the **null axis**
or B axis (glossary `null_axis`), the line where the two nodal planes meet. Roughly, P points in the direction of
the largest compression of the crust and T in the direction of the least; a single earthquake gives that
direction only roughly.

An axis is a line, not an arrow, so it is given by the end that points down: its **trend** (glossary `trend`),
the azimuth of that end clockwise from north, and its **plunge** (glossary `plunge`), the angle below the
horizontal from 0 to 90 degrees. Bulletins print the P and T axes as trend and plunge.

### M50-T3 · P and T axes · core · about 45 min

**Goal:** write `pt_axes(strike, dip, rake)`: trend and plunge of the P axis and of the T axis.

**Given:** your `fault_vectors` (task 1) and numpy. Not allowed in this function: the same libraries as in
task 1.

**Contract:**
- Inputs: `strike`, `dip`, `rake` in degrees.
- Output: `((p_trend, p_plunge), (t_trend, t_plunge))` in degrees: trends from 0 to 360 clockwise from north,
  plunges from 0 (horizontal) to 90 (vertical), each axis given by its lower end.
- Example: an invented thrust (strike 120, dip 40, rake 90) gives P = (30.0, 5.0) and T = (210.0, 85.0): P
  nearly horizontal across the strike, T nearly vertical.

**Done when** `st.check("M50-T3", func=pt_axes)` passes on five fault types and 40 random planes against ObsPy's
`mt2axes` (to 0.01 degrees). A horizontal axis may be given by either end; a vertical axis has no trend.

**Stuck?** `st.hint("M50-T3", 1)`

In [ ]:
# Before you code: where are P and T of a vertical strike-slip fault striking north (0, 90, 0)? And of the
# practice fault (30, 60, -90)? KIGAM's mechanism (Part 6) is a strike-slip on near-vertical planes striking about
# 288 and 198: roughly which way will its P axis trend, and does that fit the east-northeast to west-southwest
# compression usually given for the crust of South Korea?
st.predict("M50-T3", """

""")

In [ ]:
def pt_axes(strike, dip, rake):
    """Trend and plunge (degrees) of the P and T axes of a double-couple mechanism.

    strike, dip, rake : one nodal plane in degrees (as in fault_vectors)
    returns           : ((p_trend, p_plunge), (t_trend, t_plunge)); trend 0-360 clockwise from north,
                        plunge 0-90 below the horizontal (each axis by its lower end)
    """
    # Step 1: normal and slip (task 1); T along their sum, P along their difference (make unit vectors).
    # Step 2: make each axis point down; trend from its north and east parts, plunge from its down part.
    # YOUR CODE HERE
    raise NotImplementedError("M50-T3: write your code here")

In [ ]:
st.check("M50-T3", func=pt_axes)

Your axes for the practice faults and for KIGAM's two printed planes, next to the axes that KIGAM printed with
them (P trend 61, plunge 5; T trend 151, plunge 6; same report as the planes).

In [ ]:
for label, plane in (("vertical left-lateral", (0.0, 90.0, 0.0)), ("practice", PRACTICE_PLANE),
                     ("KIGAM plane A", KIGAM_A), ("KIGAM plane B", KIGAM_B)):
    (p_tr, p_pl), (t_tr, t_pl) = pt_axes(*plane)
    print(f"{label:22s} {str(plane):20s} P trend {p_tr:6.1f} plunge {p_pl:5.1f}   T trend {t_tr:6.1f} "
          f"plunge {t_pl:5.1f}")
print(f"{'KIGAM printed axes':22s} {'':20s} P trend {61.0:6.1f} plunge {5.0:5.1f}   T trend {151.0:6.1f} "
      f"plunge {6.0:5.1f}")

**Explain (3-5 sentences):** Were your predictions for the two practice faults right? How far are your P and T
axes of KIGAM's planes from the axes KIGAM printed, and what could cause that difference (look at how the planes
and axes are printed)? Does the P axis of the Sangju mainshock fit the regional compression you know of, and why
should one earthquake not be read as the stress of the crust?

## Part 4 · The focal sphere and its projection (about 30 min, then task 4)

Imagine a small sphere around the source, the **focal sphere** (glossary `focal_sphere`). Every ray to a
station leaves the source through one point of it: the point given by the ray's **takeoff angle** (glossary
`takeoff_angle`), measured from the downward vertical (0 straight down, 90 horizontal, 180 straight up), and its
**azimuth** (glossary `azimuth`), the direction from the source to the station, clockwise from north. The first
motion seen at the station is the first motion of the P wave at that point of the sphere.

To draw the sphere on paper, seismologists take its lower half and look at it from above: the
**lower-hemisphere projection** (glossary `lower_hemisphere_projection`). A ray straight down lands in the centre;
a horizontal ray on the rim (radius 1 here), in the direction of its azimuth, with north up and east to the
right. A ray that leaves upward (an **upgoing ray**, glossary `upgoing_ray`, takeoff above 90) is not on the lower
half; you draw the opposite direction instead, its **antipode** (glossary `antipode`): takeoff 180 − i and azimuth
+ 180. The P wave has the same sign along a direction and its opposite (it is a push or a pull along the line),
so nothing is lost.

How far from the centre? Two nets are common (glossary `stereonet`). The **equal-angle** (stereographic, Wulff)
net (glossary `equal_angle_projection`) keeps angles, so small circles stay circles, but it squeezes the area near
the centre. The **equal-area** (Lambert, Schmidt) net (glossary `equal_area_projection`) keeps areas: a patch that
covers a tenth of the half sphere covers a tenth of the circle. Seismologists use the equal-area net, so that the
number of stations in a part of the plot tells how much of the sphere they cover. The blank net you printed is an
equal-area net.

### M50-T4 · The lower-hemisphere equal-area projection · core · about 60 min

**Goal:** write `equal_area(takeoff_deg, azimuth_deg)`: where a ray lands on the lower-hemisphere equal-area
net of radius 1.

**Given:** Part 4 and numpy. Not allowed in this function: Matplotlib (all of it, with `mpl_toolkits`), the
map-projection libraries (pyproj, Cartopy, GeoPandas, PyGMT, mplstereonet), the libraries of task 1 and the
course package: several of them hold this projection.

**Contract:**
- Inputs: `takeoff_deg` (0-180 from the downward vertical) and `azimuth_deg` (clockwise from north), both one
  number, or both 1-D numpy arrays of the same length.
- Output: `(x, y)`, x to the east and y to the north, on a net of radius 1: two floats for one ray, two numpy
  arrays for arrays. Upgoing rays (takeoff above 90) are drawn at their antipode.
- Example: a horizontal ray towards an invented azimuth of 200 degrees lands on the rim at (sin 200°, cos 200°) =
  (−0.342, −0.940).

**Done when** `st.check("M50-T4", func=equal_area)` passes: the centre and the rim, the azimuth of 200 random
rays, the antipodes of 200 upgoing rays, and equal areas, counted out with 200 000 random directions.

**Stuck?** `st.hint("M50-T4", 1)`

In [ ]:
# Before you code: a ray with a takeoff of 60 degrees. On which net does it land farther from the centre, the
# equal-angle or the equal-area net, and roughly where (radius between 0 and 1)? Where does an upgoing ray with
# takeoff 120 and azimuth 30 land? Look at your hand exercise EX 3.2: where on the net did most of its stations
# fall, and why?
st.predict("M50-T4", """

""")

In [ ]:
def equal_area(takeoff_deg, azimuth_deg):
    """Position of a ray on the lower-hemisphere equal-area net of radius 1 (x east, y north).

    takeoff_deg : takeoff angle(s) from the downward vertical, 0-180 degrees (one number or a 1-D array)
    azimuth_deg : azimuth(s) clockwise from north, degrees (same shape as takeoff_deg)
    returns     : (x, y): two floats for one ray, two numpy arrays otherwise; upgoing rays at their antipode
    """
    # Step 1: upgoing rays (takeoff above 90) -> takeoff 180 - i, azimuth + 180.
    # Step 2: the radius r(i): the share of the circle's area inside r equals the share of the half sphere within
    #         takeoff i, and r(90) = 1.
    # Step 3: the point at distance r in the direction of the azimuth (x east, y north).
    # YOUR CODE HERE
    raise NotImplementedError("M50-T4: write your code here")

In [ ]:
st.check("M50-T4", func=equal_area)

Measure your net: 4000 random directions on the lower half sphere, drawn with your function and with an
equal-angle net (radius tan(i/2), written out here), and the share of the points inside half the radius on each.
Then the 60-degree ray of your prediction on both nets.

In [ ]:
rng = np.random.default_rng(50)
draws = rng.standard_normal((4000, 3))
draws[:, 2] = np.abs(draws[:, 2])
draws /= np.linalg.norm(draws, axis=1)[:, None]
rand_takeoff = np.degrees(np.arccos(draws[:, 2]))
rand_azimuth = np.degrees(np.arctan2(draws[:, 1], draws[:, 0]))
xa, ya = equal_area(rand_takeoff, rand_azimuth)
r_angle = np.tan(np.radians(rand_takeoff) / 2.0)
xs, ys = r_angle * np.sin(np.radians(rand_azimuth)), r_angle * np.cos(np.radians(rand_azimuth))
fig, axes = plt.subplots(1, 2, figsize=(8.0, 4.2))
for ax, (px, py), title in zip(axes, ((xa, ya), (xs, ys)), ("Equal-area net (yours)", "Equal-angle net")):
    ax.plot(np.cos(np.linspace(0, 2 * np.pi, 361)), np.sin(np.linspace(0, 2 * np.pi, 361)), color="0.2", lw=1.0)
    ax.plot(px, py, ".", ms=2.0, color="tab:blue")
    ax.plot(0.5 * np.cos(np.linspace(0, 2 * np.pi, 361)), 0.5 * np.sin(np.linspace(0, 2 * np.pi, 361)),
            color="tab:red", lw=1.0, ls="--", label="Half the radius")
    ax.set_aspect("equal")
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel("East")
    ax.set_ylabel("North")
    ax.set_title(title, fontsize=10)
axes[0].legend(loc="lower left", fontsize=8, frameon=False)
fig.savefig(Path(FIGURE_DIR) / "M50_random_rays_two_nets.png", dpi=150)
plt.show()
print(f"share inside half the radius: equal-area {np.mean(np.hypot(xa, ya) < 0.5):.3f}, equal-angle "
      f"{np.mean(np.hypot(xs, ys) < 0.5):.3f} (share of the circle's area inside half the radius: 0.25)")
x60, y60 = equal_area(60.0, 0.0)
print(f"takeoff 60: radius {np.hypot(x60, y60):.3f} on your net, {np.tan(np.radians(30.0)):.3f} on the "
      "equal-angle net; the upgoing ray (120, 30) lands at", np.round(equal_area(120.0, 30.0), 3))

**Explain (3-5 sentences):** Was your prediction for the 60-degree ray right? What share of the random rays falls
inside half the radius on each net, and which number shows that your net keeps areas? Why does it matter for a
first-motion plot that the net keeps areas? Where did the stations of your hand exercise EX 3.2 fall on the net,
and why there?

## Part 5 · Takeoff angles from your ray tracing (about 30 min, then task 5)

The takeoff angle of a ray comes from the velocity model. In M30 you found the direct ray to a station by
shooting: the **ray parameter** (glossary `ray_parameter`) p = sin(i) / v is the same in every layer, and you
searched for the p whose ray lands at the station. The angle at which that ray leaves the source follows from p
and the speed of the layer the source is in. Do NMSOP-2 EX 3.3 by hand before this part.

The direct ray to a surface station leaves the source upward, so its takeoff is above 90 degrees. But the
**first arrival** (glossary `first_arrival`), the wave a picker sees first and the one whose first motion you
picked, is not always the direct wave. Beyond the **crossover distance** (glossary `crossover_distance`) the
**head wave** (glossary `head_wave`) along a boundary below the source comes first; it leaves the source
downward, at the **critical angle** (glossary `critical_angle`) of that boundary, sin(i) = v_source / v_below.
Its first motion belongs to a different point of the focal sphere.

Paste your `direct_time` and `head_wave` of module M30 into the next cell as they are in your M30 notebook (they
need numpy only), and run it. Part 6 uses them to decide which wave arrives first.

In [ ]:
# Paste your direct_time and head_wave of module M30 below this line.

In [ ]:
missing = [name for name in ("direct_time", "head_wave") if not callable(globals().get(name))]
if missing:
    raise NameError(f"{' and '.join(missing)} not found: paste your functions of module M30 into the cell above "
                    "and run it (finish task M30-T2 of module M30 first; task M30-T1 writes direct_time).")
if not Path(M31_FILE).is_file():
    raise FileNotFoundError(f"{M31_FILE} is missing. This module puts the rays at your own location of the "
                            "mainshock: finish task M31-T6 of module M31 first (its cells save that table), or set "
                            "M31_FILE in the parameters cell to where you saved it.")
MODEL = st.data.velocity_model(MODEL_NAME)
located = pd.read_csv(M31_FILE).set_index("location")
SRC_LAT, SRC_LON = float(located.loc[M31_ROW, "latitude"]), float(located.loc[M31_ROW, "longitude"])
SRC_DEPTH_KM = float(located.loc[M31_ROW, "depth (km)"])
print(f"{MODEL_NAME}: {MODEL.attrs['citation']}")
print(MODEL.to_string(index=False))
print(f"\nyour location of the mainshock ({M31_ROW!r} in {M31_FILE}): {SRC_LAT:.4f} N, {SRC_LON:.4f} E, depth "
      f"{SRC_DEPTH_KM:.2f} km")

### M50-T5 · Takeoff angles from your ray tracing · core · about 60 min

**Goal:** write `takeoff_angle(x_km, depth_km, model, phase="P")`: the takeoff angle of the direct ray from a
source at `depth_km` to a surface station `x_km` away, found with your own ray tracing.

**Given:** your `direct_time` of M30 (its shooting finds the ray parameter) and numpy. Not allowed in this
function: `obspy.taup` (the check uses TauP), Pyrocko, the takeoff tables of SKHASH and PocketQuake, and the
course package.

**Contract:**
- Inputs: as `direct_time` of M30: `x_km` one number or a 1-D numpy array (≥ 0), `depth_km` (> 0), `model` a
  table like `MODEL` (not changed), `phase` `"P"` or `"S"`. A source exactly on a boundary belongs to the layer
  below.
- Output: the takeoff angle in degrees from the downward vertical (above 90 for the direct ray, 180 straight
  up), a float for one distance, a 1-D numpy array for an array, within 0.01 degrees.
- Example: in a single layer, an invented source 6 km deep and a station 6 km away give 135 degrees.

**Done when** `st.check("M50-T5", func=takeoff_angle)` passes on 4 one-layer models (geometry), 3 two-layer
models (Fermat's principle by brute force: the takeoff of the quickest path) and on kim1983, kim2011 and a random
model through TauP (to 0.1 degrees).

**Stuck?** `st.hint("M50-T5", 1)`

In [ ]:
# Before you code: your source sits at SRC_DEPTH_KM in MODEL (printed above). What takeoff angle do you expect for
# the direct P to a station 10 km away, and to one 50 km away? Beyond which distance is the head wave along the
# boundary below the source the first arrival (M30), and at what takeoff angle does it leave the source? On which
# part of the focal sphere do those stations land?
st.predict("M50-T5", """

""")

In [ ]:
def takeoff_angle(x_km, depth_km, model, phase="P"):
    """Takeoff angle (degrees from the downward vertical) of the direct ray to a surface station x_km away.

    x_km     : epicentral distance(s) in km, one number or a 1-D numpy array (>= 0)
    depth_km : source depth in km (> 0); a source on a boundary belongs to the layer below
    model    : table with top_km, vp, vs (km/s); the last row is a half-space; left unchanged
    phase    : "P" (vp) or "S" (vs)
    returns  : a float for one distance, a 1-D numpy array for an array; above 90 (the ray goes up), 180 at x = 0
    """
    # Step 1: the ray parameter p of the direct ray that lands at x_km (shooting, as in your direct_time).
    # Step 2: the speed of the source layer; the angle from the vertical there has sin = p * v_source.
    # Step 3: the ray goes up: takeoff = 180 degrees minus that angle.
    # YOUR CODE HERE
    raise NotImplementedError("M50-T5: write your code here")

In [ ]:
st.check("M50-T5", func=takeoff_angle)

Your takeoff angles of the direct P and S at your source depth, the crossover distance of every crustal boundary
below the source from your `head_wave` (per phase; NaN where no head wave can run), and the critical angle at
which each head wave leaves the source.

In [ ]:
x_show = np.array([0.0, 5.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 80.0, 100.0])
print(pd.DataFrame({"x_km": x_show, "direct P takeoff": takeoff_angle(x_show, SRC_DEPTH_KM, MODEL, "P"),
                    "direct S takeoff": takeoff_angle(x_show, SRC_DEPTH_KM, MODEL, "S")}).round(2)
      .to_string(index=False))
tops = MODEL["top_km"].to_numpy(float)
for phase, column in (("P", "vp"), ("S", "vs")):
    speeds = MODEL[column].to_numpy(float)
    v_source = speeds[tops <= SRC_DEPTH_KM][-1]
    for n in range(1, len(MODEL) - 1):              # crustal boundaries; the Moho (Pn, Sn) is left out
        _, x_crit, x_cross = head_wave(np.array([50.0]), SRC_DEPTH_KM, MODEL, phase, n)
        if np.isfinite(x_cross):
            print(f"{phase} along {tops[n]:g} km: critical distance {x_crit:.1f} km, crossover {x_cross:.1f} km, "
                  f"leaves the source at {np.degrees(np.arcsin(v_source / speeds[n])):.1f} degrees (downward)")
        else:
            print(f"{phase} along {tops[n]:g} km: no head wave from a source at {SRC_DEPTH_KM:.2f} km")

**Explain (3-5 sentences):** Were your predicted takeoff angles at 10 and 50 km right? Beyond which distance
does a head wave come first at your source depth, and at what takeoff angle does it leave? A small change of
your source depth moves that crossover a lot (M30): what would a station just beyond it do to a first-motion plot
if you used the direct-wave takeoff by mistake? Why does the takeoff of S not matter for P first motions?

## Part 6 · The Sangju first motions on the focal sphere (about 40 min, then task 6)

Now the first motions you picked in M20 for the ML 3.9 Sangju mainshock. Each clear P first motion is a
**compression** (glossary `compression`, U: the ground first moved up, pushed away from the source) or a
**dilatation** (glossary `dilatation`, D: pulled towards it). Your polarities are what you saw in the counts.
M03 (Part 6) showed that in the station metadata of the course (StationMetadata 1.0.2 by Byeong-Seok Ahn,
reference `ahn2024`) a negative first-stage gain is not a **polarity reversal** (glossary `polarity_reversal`),
so here no polarity is ever flipped by a sign in the metadata.

The mechanism to test is the one of the Korea Institute of Geoscience and Mineral Resources (KIGAM) in its 2019
bulletin (*2019 학술지진연보*, KIGAM 2021, report KEMS-2019-062; reference `kigam2021_bulletin2019`): nodal planes
A = (288, 89, 7) and B = (198, 83, 179), strike, dip and rake in degrees, P axis 61/5 and T axis 151/6 (trend and
plunge). The bulletin does not say how the mechanism was found.

Which first motion does a mechanism predict? The P wave along a ray γ (a unit vector, north-east-down, from the
takeoff and the azimuth) has the amplitude 2 (γ · n)(γ · d) for a unit fault (Aki & Richards 2002, chapter 4;
this is the P **radiation pattern**, glossary `radiation_pattern`). Positive means a push: compression, U. So the
first motion changes sign wherever the ray crosses one of the two nodal planes (γ · n = 0 or γ · d = 0), and a
station near a nodal plane can show either sign.

The next cells read your picks, your location (Part 5) and the station coordinates from the SAC headers of your
download, and keep the stations with a clear P first motion (U or D). A helper draws a focal sphere with your
own functions: the quadrants where your `polarity_agreement` (task 6) predicts compression, grey, with the nodal
planes as their edges; your P and T axes; and the first motions where your `equal_area` puts them (filled: U,
open: D).

In [ ]:
catalog = st.data.kma_catalog()
event = catalog.loc[catalog["event_id"] == KMA_EVENT_ID].iloc[0]
ORIGIN_KMA = UTCDateTime(event["origin_utc"].isoformat())
SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / ORIGIN_KMA.strftime("%Y%m%d%H%M%S")
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no waveforms of this event in {SAC_DIR}. Fetch the {DATASET} data with STP "
                            "first (module M03), or set DATASET in the parameters cell to the folder you used.")
if not Path(PICKS_CSV).is_file():
    raise FileNotFoundError(f"{PICKS_CSV} is missing. This module plots your own first motions: finish task M20-T5 "
                            "of module M20 first (its picker saves the file).")
print(f"KMA {KMA_EVENT_ID}: origin {ORIGIN_KMA} (UTC), {float(event['latitude']):.2f} N "
      f"{float(event['longitude']):.2f} E, depth {float(event['depth_km']):g} km, ML {float(event['ml']):.1f}; "
      f"source of the catalog: {catalog.attrs.get('source', 'KMA')}")
picks = pd.read_csv(PICKS_CSV, dtype={"station": str, "phase": str, "polarity": str})
p_picks = picks[picks["phase"] == "P"].drop_duplicates("station")
rows = []
for name, first_motion in zip(p_picks["station"], p_picks["polarity"].fillna("?").str.strip()):
    if first_motion not in ("U", "D"):
        continue
    net, sta = name.split(".")
    header = next(iter(sorted(SAC_DIR.glob(f"*/*.{net}.{sta}.*Z.sac"))), None)
    if header is None:
        raise FileNotFoundError(f"No vertical record of {name} in {SAC_DIR}: check the station name in {PICKS_CSV}.")
    sac = read(str(header), headonly=True)[0].stats.sac
    rows.append({"station": name, "latitude": float(sac.stla), "longitude": float(sac.stlo),
                 "polarity": first_motion})
fm_stations = pd.DataFrame(rows)
print(f"\n{len(fm_stations)} stations with a clear P first motion in {PICKS_CSV}:")
print(fm_stations.to_string(index=False))

In [ ]:
def draw_focal_sphere(ax, plane, project, agreement, axes=None, table=None, title=""):
    """Draw a lower-hemisphere focal sphere with YOUR functions (nothing here computes a mechanism).

    plane     : (strike, dip, rake) in degrees
    project   : your equal_area (task 4); places every point
    agreement : your polarity_agreement (task 6); its predicted first motions shade the compressional quadrants
                (grey), and the edges of the grey areas are the nodal planes
    axes      : your pt_axes (task 3), or None; marks P and T
    table     : a DataFrame with takeoff_deg, azimuth_deg and polarity (U filled, D open), or None
    """
    grid_takeoff, grid_azimuth = np.meshgrid(np.linspace(0.0, 90.0, 181), np.linspace(0.0, 360.0, 721),
                                             indexing="ij")             # every 0.5 degree of the lower half
    predicted, _ = agreement(grid_takeoff.ravel(), grid_azimuth.ravel(), ["?"] * grid_takeoff.size, *plane)
    gx, gy = project(grid_takeoff.ravel(), grid_azimuth.ravel())
    gx, gy = np.reshape(gx, grid_takeoff.shape), np.reshape(gy, grid_takeoff.shape)
    push = np.where(np.asarray(predicted) == "U", 1.0, -1.0).reshape(grid_takeoff.shape)
    ax.contourf(gx, gy, push, levels=[-2.0, 0.0, 2.0], colors=["white", "0.72"])
    ax.contour(gx, gy, push, levels=[0.0], colors="k", linewidths=1.0)
    rim = np.linspace(0.0, 360.0, 361)
    turn = np.radians(rim)
    ax.plot(np.sin(turn), np.cos(turn), color="k", lw=1.2)
    ax.plot([0.0], [0.0], "+", color="k", ms=6)
    ax.text(0.0, 1.04, "N", ha="center", va="bottom", fontsize=9)
    if axes is not None:
        for label, (trend, plunge) in zip(("P", "T"), axes(*plane)):
            px, py = project(90.0 - plunge, trend)
            ax.text(px, py, label, ha="center", va="center", fontsize=11, fontweight="bold",
                    color="tab:red" if label == "P" else "tab:blue")
    if table is not None and len(table):
        tx, ty = project(table["takeoff_deg"].to_numpy(float), table["azimuth_deg"].to_numpy(float))
        ups = table["polarity"].to_numpy() == "U"
        ax.plot(tx[ups], ty[ups], "o", ms=8, mfc="k", mec="k", label="U (compression)")
        ax.plot(tx[~ups], ty[~ups], "o", ms=8, mfc="white", mec="k", mew=1.5, label="D (dilatation)")
        if "station" in table:
            for sx, sy, name in zip(tx, ty, table["station"]):
                ax.text(sx + 0.04, sy + 0.04, name.split(".")[-1], fontsize=7)
    ax.set_aspect("equal")
    ax.set_xlim(-1.12, 1.12)
    ax.set_ylim(-1.12, 1.12)
    ax.axis("off")
    ax.set_title(title, fontsize=10)

### M50-T6 · First motions of the Sangju mainshock on the focal sphere · core · about 120 min

**Goal:** write `polarity_agreement(takeoff_deg, azimuth_deg, polarity, strike, dip, rake)`, then put your
Sangju first motions on the focal sphere with KIGAM's mechanism, save the table and the figure, and fill in the
dictionary `sangju_fm`.

**Given:** your functions of tasks 1-5, your M30 functions (Part 5), `fm_stations`, `SRC_LAT`, `SRC_LON`,
`SRC_DEPTH_KM`, `MODEL`, `KIGAM_A`, `draw_focal_sphere`, and `gps2dist_azimuth`. Not allowed in
`polarity_agreement`: the same libraries as in task 1.

**Contract (the function):**
- Inputs: `takeoff_deg`, `azimuth_deg` (1-D numpy arrays, degrees, as in task 4), `polarity` (a list or array of
  `"U"`, `"D"` or `"?"`, one per ray), and one nodal plane `strike`, `dip`, `rake` (degrees).
- Output: `(predicted, fraction)`: `predicted` a numpy array of `"U"` or `"D"` (the first motion that the
  mechanism predicts along each ray), and `fraction` the share of the clear first motions (U or D) that equal the
  prediction, a float (NaN when there is no clear first motion).
- Example: three invented rays with the polarities U, D, ? and predictions U, U, D give the fraction 1/2.

**Contract (the files and the answer):**
- `results/M50_first_motions.csv`, one row per station of `fm_stations`, with the columns `station`, `latitude`,
  `longitude`, `distance_km` (epicentral, from your source), `azimuth_deg` (from your source to the station),
  `wave` (`"direct"` or `"head"`: the P first arrival from your M30 functions, crustal boundaries only),
  `takeoff_deg` (of that wave: your `takeoff_angle` for a direct wave, the critical angle for a head wave),
  `polarity` (as picked) and `predicted` (from your function, for KIGAM's plane A).
- `figures/M50_sangju_focal_sphere.png`: your focal sphere of KIGAM's plane A with your first motions
  (`draw_focal_sphere`, or a figure of your own).
- `sangju_fm = {"agreement": polarity_agreement, "table": <CSV path>, "png": <PNG path>, "picks": PICKS_CSV,
  "mechanism": KIGAM_A, "source": {"latitude": SRC_LAT, "longitude": SRC_LON, "depth_km": SRC_DEPTH_KM},
  "model": MODEL_NAME, "fraction": <your fraction for the table>}`. The ObsPy cell after your code adds one
  more key, `"obspy_runs"`: the table of its three runs of ObsPy's beachball.

**Done when** `st.check("M50-T6", answer=sangju_fm)` passes: your function on 10 made-up sets of rays first, then
your files: your polarities, the distances, azimuths and coordinates, the first arrivals and takeoffs (TauP), the
predictions (Pyrocko) and the fraction, all recomputed by the check from your table; last the three ObsPy runs,
redrawn and recomputed from their files. It prints no value.

**Stuck?** `st.hint("M50-T6", 1)`

In [ ]:
# Before you code: look at KIGAM's planes (288/89/7 and 198/83/179) and at the azimuths of your stations. How many
# of your clear first motions do you expect to agree with the mechanism, and which stations could disagree? Which
# of your stations have a head wave as their first arrival, and where will they land on the net compared with the
# others? After your code, a cell draws the mechanism with ObsPy's beachball, from plane A twice and then from
# plane B (CHANGED_PLANE): what will change in the picture?
st.predict("M50-T6", """

""")

In [ ]:
def polarity_agreement(takeoff_deg, azimuth_deg, polarity, strike, dip, rake):
    """Predicted P first motions of a mechanism along rays, and the share of observed ones that agree.

    takeoff_deg, azimuth_deg : 1-D numpy arrays (degrees; takeoff from the downward vertical, azimuth clockwise
                               from north)
    polarity                 : observed first motions, "U", "D" or "?" (one per ray; "?" is not counted)
    strike, dip, rake        : one nodal plane (degrees)
    returns                  : (predicted, fraction): a numpy array of "U"/"D", and the share of the clear first
                               motions (U or D) that equal the prediction (NaN when there is none)
    """
    # Step 1: the normal and the slip of the plane (task 1).
    # Step 2: one unit ray per station (north, east, down) from its takeoff and azimuth.
    # Step 3: U where the ray's parts along the normal and along the slip have the same sign, else D.
    # Step 4: the share of the clear first motions that equal the prediction.
    # YOUR CODE HERE
    raise NotImplementedError("M50-T6: write your code here")


# Build the table of the Contract, save it and the figure, and fill in sangju_fm, in the steps below.
# Step 1: the P speeds and the layer tops of MODEL, and the P speed of the layer your source is in.
# YOUR CODE HERE
raise NotImplementedError("M50-T6: write your code here")

# Step 2: for each station of fm_stations: the epicentral distance in km and the azimuth from your source to the
#         station, clockwise from north (gps2dist_azimuth; read its documentation for what it returns, in which
#         order and in which unit).
# Step 3: the wave that arrives first at that distance: the time of the direct P wave (your direct_time) against
#         the head wave along each crustal boundary below the source (your head_wave; never the Moho, no Pn).
#         The earlier time names the wave: "direct" or "head".
# Step 4: the takeoff angle of that wave, from the downward vertical: your takeoff_angle (task 5) for a direct
#         wave; for a head wave the critical angle of its boundary, sin(i) = v_source / v_below (Part 5).
#         One row per station: name, coordinates, distance, azimuth, wave, takeoff and polarity as you picked it.
# YOUR CODE HERE
raise NotImplementedError("M50-T6: write your code here")

# Step 5: the table, sorted by distance, with the first motions your polarity_agreement predicts for KIGAM's
#         plane A in the column predicted, and the fraction it returns; save the table as CSV in RESULT_DIR.
# YOUR CODE HERE
raise NotImplementedError("M50-T6: write your code here")

# Step 6: the figure: draw_focal_sphere puts every ray on the net with your equal_area and shades the quadrants
#         with your polarity_agreement; save it as PNG in FIGURE_DIR.
# YOUR CODE HERE
raise NotImplementedError("M50-T6: write your code here")

# Step 7: fill in sangju_fm with the keys of the Contract.
# YOUR CODE HERE
raise NotImplementedError("M50-T6: write your code here")

**The production tool.** ObsPy's `beach` draws a mechanism as seismologists print it: a **beachball** (glossary
`beachball`), the lower-hemisphere equal-area projection with the compressional quadrants filled. It reads a nodal
plane (strike, dip, rake) or a moment tensor in up-south-east order. The next cell runs it three times with the
function `obspy_ball`: KIGAM's plane A, the same again unchanged, and then the plane named by `CHANGED_PLANE`,
the one input you change. It saves each picture as a PNG file with its **checksum** (glossary `checksum`, the
SHA-256 fingerprint of the file), counts how many pixels differ from the first picture and how many are grey
(the fill colour of the quadrants), and writes the three runs to `results/M50_obspy_runs.csv`. It adds that path
to `sangju_fm` as `obspy_runs`: the check of task 6 redraws ObsPy's beachball of each plane and recomputes every
number of the runs table from your files. Run it after your code above and before the check; it prints the
runs only after the check.

In [ ]:
import matplotlib
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from obspy.imaging.beachball import beach
from PIL import Image


def obspy_ball(plane, table=None, png=None):
    """ObsPy's beachball of one nodal plane, drawn alone with Matplotlib's default settings (so the picture does
    not depend on the settings of this session); saved as png (100 dots per inch) when a path is given."""
    with matplotlib.style.context("default"):
        ball_fig = Figure(figsize=(3.0, 3.0), dpi=100)
        FigureCanvasAgg(ball_fig)
        ball_ax = ball_fig.add_axes([0.0, 0.0, 1.0, 1.0])
        ball_ax.add_collection(beach(list(plane), xy=(0.0, 0.0), width=2.0, linewidth=1.0, facecolor="0.55"))
        if table is not None:
            tx, ty = equal_area(table["takeoff_deg"].to_numpy(float), table["azimuth_deg"].to_numpy(float))
            ups = table["polarity"].to_numpy() == "U"
            ball_ax.plot(tx[ups], ty[ups], "o", ms=7, mfc="k", mec="k")
            ball_ax.plot(tx[~ups], ty[~ups], "o", ms=7, mfc="white", mec="k", mew=1.5)
        ball_ax.set_xlim(-1.05, 1.05)
        ball_ax.set_ylim(-1.05, 1.05)
        ball_ax.set_aspect("equal")
        ball_ax.axis("off")
        if png is not None:
            ball_fig.savefig(png, dpi=100)
    return ball_fig


def png_pixels(path):
    """The pixels (RGBA, 0-255) of a PNG file, as the file holds them."""
    with Image.open(path) as image:
        return np.asarray(image.convert("RGBA"), dtype=np.uint8).copy()


planes = {"A": KIGAM_A, "B": KIGAM_B}
obspy_rows = []
for run, label in (("unchanged 1", "A"), ("unchanged 2", "A"), ("changed", CHANGED_PLANE)):
    ball_png = Path(FIGURE_DIR) / f"M50_obspy_{run.replace(' ', '_')}.png"
    obspy_ball(planes[label], png=ball_png)
    pixels = png_pixels(ball_png)
    if not obspy_rows:
        first_pixels = pixels
    gap = np.abs(first_pixels.astype(int) - pixels.astype(int)).max(axis=2)    # largest change of a pixel
    grey = np.all(np.abs(pixels[..., :3].astype(int) - 140) <= 3, axis=2)      # the fill colour, 0.55 of 255
    obspy_rows.append({"run": run, "plane": label, "strike": planes[label][0], "dip": planes[label][1],
                       "rake": planes[label][2], "png": str(ball_png),
                       "sha256": hashlib.sha256(ball_png.read_bytes()).hexdigest(), "pixels": int(gap.size),
                       "pixels_changed": int((gap > 0).sum()), "strongly_changed": int((gap > 64).sum()),
                       "grey_pixels": int(grey.sum())})
runs_path = Path(RESULT_DIR) / "M50_obspy_runs.csv"
pd.DataFrame(obspy_rows).to_csv(runs_path, index=False)
sangju_fm["obspy_runs"] = str(runs_path)      # the check of task 6 reads the three runs from this table
print(f"wrote {len(obspy_rows)} pictures of ObsPy's beachball to {FIGURE_DIR}/ and their table to {runs_path}")

In [ ]:
st.check("M50-T6", answer=sangju_fm)

Your table, and for every station the smallest turn of its ray (in degrees, by steps of 0.5, up to 45) that
flips the first motion your function predicts: a small number means a station close to a nodal plane, NaN one
that no turn up to 45 degrees flips.

In [ ]:
shown = pd.read_csv(sangju_fm["table"])
base = shown["predicted"].to_numpy()
margin = np.full(len(shown), np.nan)
for step in np.arange(0.5, 45.01, 0.5):
    for d_take, d_az in ((step, 0.0), (-step, 0.0), (0.0, step), (0.0, -step)):
        turned, _ = polarity_agreement(np.clip(shown["takeoff_deg"].to_numpy() + d_take, 0.0, 180.0),
                                       shown["azimuth_deg"].to_numpy() + d_az, ["?"] * len(shown), *KIGAM_A)
        margin[np.isnan(margin) & (turned != base)] = step
shown["agree"] = shown["polarity"] == shown["predicted"]
shown["flip margin (deg)"] = margin
print(shown.drop(columns=["latitude", "longitude"]).round(2).to_string(index=False))
print(f"\n{int(shown['agree'].sum())} of {len(shown)} clear first motions agree with KIGAM's plane A "
      f"(fraction {sangju_fm['fraction']:.3f}); head waves first at {int((shown['wave'] == 'head').sum())} stations")

Your three runs of ObsPy's beachball (the table the check has just recomputed; the checksums cut to 16
characters), and your first motions (your `equal_area`) on ObsPy's ball of plane A.

In [ ]:
runs_table = pd.read_csv(sangju_fm["obspy_runs"])
runs_table["sha256"] = runs_table["sha256"].str[:16]
runs_table["changed %"] = 100.0 * runs_table["pixels_changed"] / runs_table["pixels"]
runs_table["grey %"] = 100.0 * runs_table["grey_pixels"] / runs_table["pixels"]
print(runs_table.drop(columns=["png"]).round(2).to_string(index=False))
with_points = obspy_ball(KIGAM_A, shown, png=Path(FIGURE_DIR) / "M50_obspy_beachball_plane_A.png")
with_points

**Explain (3-5 sentences):** How many of your clear first motions agree with KIGAM's mechanism? Which stations
disagree, how close are they to a nodal plane (their flip margin), and what else could make a first motion
disagree (think of the pick itself, the location and depth, and the velocity model)? Where did the stations whose
first arrival is a head wave land, compared with the direct-wave stations? What did ObsPy draw for the unchanged
rerun and for plane B, and what does that say about which plane slipped?

Optional explorer E13 (not checked): turn strike, dip and rake and watch your quadrants move over your first
motions. Everything it draws comes from your functions: the shading and the count from `polarity_agreement`, the
points from `equal_area`, P and T from `pt_axes`, the second plane from `auxiliary_plane`. KIGAM's plane A is the
dashed reference.

In [ ]:
st.widgets.focal_sphere_fitter(shown, project=equal_area, agreement=polarity_agreement, axes=pt_axes,
                               aux_plane=auxiliary_plane, strike=KIGAM_A[0], dip=KIGAM_A[1], rake=KIGAM_A[2],
                               reference=KIGAM_A)

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M50")

### Going further (optional, not checked)

1. In the explorer E13, find the range of strikes of plane A (keeping dip and rake) for which all your first
   motions agree. How well do your stations fix the mechanism?
2. Set `M31_ROW` to HYPOINVERSE's location (`"run 1"`) or use the KMA depth (14 km): which stations change from
   head wave to direct wave, and does the agreement change?
3. Redo Part 6 with `MODEL_NAME = "kim2011"`. The source then sits in a different layer: what happens to the
   takeoff angles and to the head waves?
4. If you ran PocketQuake with PhaseNet+ in M21, add its first motions of the mainshock (M21 Part 5) for stations
   you did not pick, and draw them on the same sphere.

Next: M51 finds the mechanism from first motions by a grid search and with SKHASH, and compares it with KIGAM's.